# Module 3.2: Multi-Head Attention & Causal Masking

In **Module 3.1** you built a single attention head and watched `bank` attend to
`river`. Two things were still missing, and this notebook adds both:

1. **Multi-head attention** — one head has to settle on a single compromise notion of
   "relevant". Real models run 8, 32, or 64 heads in parallel so different heads can
   specialise in different relationships.
2. **Causal masking** — right now every word can see every other word, including the
   ones that come *after* it. For a model that generates text left to right, that's
   letting it peek at the answer.

We finish with the bill: **what attention actually costs**, which is the problem the
whole of Part 7 exists to solve.

## 0. Setup — the pieces from Module 3.1

Every notebook in this course runs standalone, so we re-create the two things we need
from 3.1: the `scaled_dot_product_attention` function and the hand-crafted
`bank`/`river` embeddings.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from math import sqrt

torch.manual_seed(0)

def scaled_dot_product_attention(query, key, value, mask=None):
    """The four stages from Module 3.1, unchanged."""
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        # Where the mask is 0 ("not allowed to look here"), set the score to a huge
        # negative number. After softmax, exp(-1e9) is effectively 0.
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attention_weights, value), attention_weights

# The hand-crafted sentence from Module 3.1: dims are roughly
# [function_word, finance, nature, water].
words = ["the", "bank", "of", "the", "river"]
emb = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # the
    [0.0, 0.3, 0.9, 0.2],   # bank
    [1.0, 0.0, 0.0, 0.0],   # of
    [1.0, 0.0, 0.0, 0.0],   # the
    [0.0, 0.0, 0.9, 0.9],   # river
]).unsqueeze(0)             # -> (1, 5, 4)

# A batch of random embeddings for shape-tracking demos: 1 sentence, 5 words, 128 dims.
dummy_sentence_embeddings = torch.randn(1, 5, 128)
print("emb:", tuple(emb.shape), "| dummy:", tuple(dummy_sentence_embeddings.shape))

## 3. Multi-Head Attention (The Committee of Experts)

### The Analogy
Imagine reviewing a legal contract. If you review it alone, you might focus on the grammar but miss financial loopholes. 
Now imagine evaluating the contract with a **Committee of 8 Experts**. Expert 1 only checks grammar. Expert 2 only checks finances. Expert 3 tracks pronouns. 

**Multi-Head Attention (MHA)** is exactly this. Instead of one giant Attention layer, we run several smaller "Heads" in parallel and glue their findings back together at the end.

### A common misconception (important!)
We do **NOT** slice the raw embedding into 8 chunks and give each head a different slice of the original word vector. Instead:
1. One big `W_q` (and `W_k`, `W_v`) projects the **whole** `d_model`-dimensional embedding into a new `d_model`-dimensional space.
2. We then **reshape** that projection into `num_heads` groups of `d_k` dimensions each (e.g. 128 → 8 × 16).

So every head's slice comes *after* a learned projection that already mixed information from the entire embedding. Each head therefore draws on the whole word, just through a different learned "lens" — not from a fixed raw segment.

### Why do we need it? (Specialization)
Language is incredibly nuanced. If we only had one Single Head, the gradients would force the model to compromise and try to find a "middle ground" of attention. MHA lets the network look at a single word from 8 (or 16, or 32) different, specialized conceptual angles at once without that compromise.

### The reshape, one axis at a time

This is the step that trips up more people than anything else in Transformer code, so
let's walk it slowly. You already met 4-D tensors in **Module 0.1 §6** — this is where
that pays off.

We start with the projected queries, shape `(B, T, d_model)`:

```
Q = self.W_q(x)                                  (1, 5, 128)
                                                  B  T   d_model
```

**Step 1 — `.view(B, T, num_heads, d_k)`**: split the last axis into a grid. Nothing
moves in memory; we just *reinterpret* 128 numbers as 8 groups of 16.

```
(1, 5, 128)  ->  (1, 5, 8, 16)
                  B  T  H   d_k
```

**Step 2 — `.transpose(1, 2)`**: swap the token axis and the head axis, so that each
head owns a contiguous `(T, d_k)` matrix to attend over.

```
(1, 5, 8, 16)  ->  (1, 8, 5, 16)
 B  T  H  d_k       B  H  T  d_k
```

Read the final shape out loud: **"1 sentence, 8 heads, 5 tokens each, 16 numbers per
token."** Now `scaled_dot_product_attention` runs on all 8 heads *at once*, because
`@` broadcasts over the leading `(B, H)` axes — the batched matmul from Module 0.1.

Afterwards we undo it exactly in reverse to glue the heads back together:

```
(1, 8, 5, 16)  --transpose(1,2)-->  (1, 5, 8, 16)  --view-->  (1, 5, 128)
```

> **Why `.contiguous()` before that last `.view()`?** `transpose` doesn't move data,
> it just relabels the axes — so the tensor's memory layout no longer matches its
> shape, and `.view()` (which requires a compatible layout) refuses. `.contiguous()`
> makes a properly-ordered copy. If you ever see *"view size is not compatible with
> input tensor's size and stride"*, this is the fix.

Let's watch every one of those shapes print:

In [ ]:
# The reshape dance, printed step by step -- no attention math, just shapes.
B, T, d_model, H = 1, 5, 128, 8
d_k = d_model // H

x = torch.randn(B, T, d_model)
print(f"start                      {tuple(x.shape)}   (B, T, d_model)")

step1 = x.view(B, T, H, d_k)
print(f"after .view(B,T,H,d_k)     {tuple(step1.shape)}  (B, T, H, d_k)")

step2 = step1.transpose(1, 2)
print(f"after .transpose(1,2)      {tuple(step2.shape)}  (B, H, T, d_k)  <- attention runs here")

step3 = step2.transpose(1, 2)
print(f"back .transpose(1,2)       {tuple(step3.shape)}  (B, T, H, d_k)")

step4 = step3.contiguous().view(B, T, d_model)
print(f"back .view(B,T,d_model)    {tuple(step4.shape)}   (B, T, d_model)")

print("\nRound-trip is lossless:", torch.equal(x, step4))
print("d_k =", d_k, "-> 8 heads x 16 dims = 128, the full d_model. No information is dropped.")

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head (e.g., 128 / 8 = 16)

        # bias=False, same reasoning as the single-head version.
        # Each of these projects the FULL d_model embedding -> d_model (not a slice).
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False) # The "Glue" at the end

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()

        # 1. Linear Projections (each looks at the WHOLE embedding)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. RESHAPE the projected vectors into heads (we are NOT slicing the raw
        #    embedding -- W_q already mixed the whole vector; .view just regroups it).
        # (Batch, Seq_Len, d_model) -> (Batch, Seq_Len, Num_Heads, Head_Dim) -> (Batch, Num_Heads, Seq_Len, Head_Dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 3. Apply Attention (Parallel over all heads!)
        output, _ = scaled_dot_product_attention(Q, K, V, mask=mask)

        # 4. Glue the Heads back together
        # Transpose back: (Batch, Num_Heads, Seq_Len, Head_Dim) -> (Batch, Seq_Len, Num_Heads, Head_Dim)
        # Contiguous/View: smashes Num_Heads and Head_Dim back into d_model
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        # 5. Final Output Projection
        return self.W_o(output)

# Look at the Dimensionality Tracking!
mha = MultiHeadAttention(d_model=128, num_heads=8)
final_output = mha(dummy_sentence_embeddings)

print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {final_output.shape}")

## 4. Causal Masking (No Peeking at the Future)

So far every word could look at every other word — including words that come *later* in the sentence. That is fine for tasks that read a whole sentence at once (like classification). But an LLM is **autoregressive**: it generates text one token at a time, predicting the next word from the words so far. During training we feed the whole sentence at once for speed, so we must stop each position from "cheating" by looking at the answer (the words that come after it).

The fix is a **causal mask**: a lower-triangular matrix of 1s. Position `i` is only allowed to attend to positions `0..i` (itself and everything before it). Wherever the mask is 0 (the upper triangle = the future), our `scaled_dot_product_attention` sets the score to a huge negative number, so after softmax those future positions get ~zero weight.

Let's watch the upper triangle of the attention matrix collapse to 0.

In [ ]:
# Build a causal (lower-triangular) mask for a length-5 sequence.
# 1 = "allowed to look here", 0 = "blocked (it's in the future)".
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("Causal mask (1 = allowed, 0 = blocked future):")
print(causal_mask.int())

# Reuse the hand-crafted "bank/river" embeddings from earlier, now WITH the mask.
_, masked_attn = scaled_dot_product_attention(emb, emb, emb, mask=causal_mask)
masked_attn = masked_attn[0]

print("\nMasked attention weights (upper triangle should be 0):\n")
header = "          " + "".join(f"{w:>8}" for w in words)
print(header)
for i, w in enumerate(words):
    row = "".join(f"{masked_attn[i, j]:8.2f}" for j in range(len(words)))
    print(f"{w:>10}{row}")

# The very first word can only attend to itself; each later row opens up one more column.
print("\nUpper-triangle (future) total weight, should be ~0:",
      round(masked_attn.triu(diagonal=1).sum().item(), 6))
print("Every row still sums to 1.0:", [round(v, 3) for v in masked_attn.sum(-1).tolist()])

plt.figure(figsize=(5, 4))
plt.imshow(masked_attn.numpy(), cmap="viridis")
plt.xticks(range(len(words)), words)
plt.yticks(range(len(words)), words)
plt.xlabel("Looking AT")
plt.ylabel("Querying word")
plt.title("Causal (masked) attention weights")
plt.colorbar()
plt.show()

## 5. The bill: what attention costs

Attention gave us something RNNs never had — any token can reach any other token in a
**single** step, with no forgetting. That power has a price, and the price is the
reason Part 7 of this course exists.

Look again at the shape of the thing we just printed: the attention-weight matrix is
`(T, T)` — one number for **every pair of tokens**. So for a sequence of length $T$:

- **Compute** is $O(T^2 \cdot d)$ — the $QK^T$ matmul touches every pair.
- **Memory** is $O(T^2)$ per head, per layer, just to hold the weights.

Doubling the context length **quadruples** both. That single fact drives an enormous
amount of modern LLM engineering:

| Consequence | Where we deal with it |
|---|---|
| Generating token-by-token re-computes the same keys/values over and over | **7.1 KV caching** |
| The cache itself becomes the memory bottleneck | **7.2 MQA & GQA** |
| The $(T,T)$ matrix is too big to even write to GPU memory | **7.3 FlashAttention** |
| Serving many users means packing caches efficiently | **9.3 PagedAttention** |

Let's make the quadratic real — because "quadratic" is an abstraction until you see
the numbers.

In [ ]:
# How big is the attention matrix, really? (one layer, one head, fp16 = 2 bytes)
print(f"{'context T':>10} {'pairs (T^2)':>15} {'weights, 32 layers x 32 heads':>32}")
print("-" * 60)
for T in [512, 2_048, 8_192, 32_768, 131_072]:
    pairs = T * T
    # 32 layers x 32 heads x T*T weights x 2 bytes, in GB
    gb = 32 * 32 * pairs * 2 / 1e9
    print(f"{T:>10,} {pairs:>15,} {gb:>29,.1f} GB")

print("\nThat last row is a 128K context -- and no GPU on earth has that much memory.")
print("FlashAttention (7.3) works precisely because it NEVER materialises the T x T matrix.")

## Summary

You now have the complete attention block that every Transformer uses:

- **Multi-head attention** projects the whole embedding with `W_q`/`W_k`/`W_v`, reshapes
  it into `H` parallel heads, attends within each, glues them back with `.contiguous()
  .view()`, and mixes them with a final `W_o`. Each head sees the *whole* word through
  a different learned lens — it is not a slice of the raw embedding.
- **Causal masking** zeroes the upper triangle so position `i` can only attend to
  `0..i`. This is the one line that makes a Transformer *generative*.
- **The cost is quadratic in sequence length**, which is the problem Part 7 solves.

Next, in **Module 4.1**, we wrap this block in the machinery that makes it trainable
at depth: residual connections, normalization, and the feed-forward network.

### 🏋️ Try it yourself

1. **Read one head's mind.** `MultiHeadAttention` throws its attention weights away.
   Modify `forward()` to stash them (`self.last_weights = w`), run
   `MultiHeadAttention(d_model=4, num_heads=2)` on `emb`, and print head 0 and head 1
   side by side. Do the two heads attend differently?
2. **Confirm the mask survives the heads.** Pass `mask=causal_mask` through the
   multi-head layer and verify the upper triangle of `last_weights` is ~0 for *every*
   head, not just the first.
3. **Find the crossover.** Time `scaled_dot_product_attention` on random tensors at
   `T = 128, 256, 512, 1024, 2048` and plot time against `T`. Confirm the curve bends
   upward like $T^2$ rather than climbing in a straight line.

In [ ]:
# Task 2 starter: the mask threads through the multi-head layer unchanged.
mha_small = MultiHeadAttention(d_model=4, num_heads=2)
masked_out = mha_small(emb, mask=causal_mask)
print("Output shape with causal mask:", masked_out.shape)

# Task 1 hint: inside MultiHeadAttention.forward, replace
#     output, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
# with
#     output, w = scaled_dot_product_attention(Q, K, V, mask=mask)
#     self.last_weights = w          # (B, H, T, T)
# then inspect mha_small.last_weights[0, 0] and [0, 1].

# Task 3 starter:
import time
for T in [128, 256, 512, 1024]:
    q = torch.randn(1, 8, T, 64)
    t0 = time.perf_counter()
    scaled_dot_product_attention(q, q, q)
    print(f"T={T:>5}  {(time.perf_counter() - t0) * 1000:7.1f} ms")